In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# u.data 로드 (탭 구분, 컬럼명 직접 지정)
ratings = pd.read_csv('../data/raw/u.data', sep='\t',
                      names=['userId', 'movieId', 'rating', 'timestamp'])

# u.item 로드 (영화 정보, 파이프 구분)
movies = pd.read_csv('../data/raw/u.item', sep='|', encoding='latin-1',
                     names=['movieId','title','release_date','video_release',
                            'imdb_url','unknown','Action','Adventure','Animation',
                            'Children','Comedy','Crime','Documentary','Drama',
                            'Fantasy','Film-Noir','Horror','Musical','Mystery',
                            'Romance','Sci-Fi','Thriller','War','Western'],
                     usecols=['movieId','title','release_date'])

print("ratings shape:", ratings.shape)
print(ratings.head())
print("\nmovies shape:", movies.shape)
print(movies.head())

ratings shape: (100000, 4)
   userId  movieId  rating  timestamp
0     196      242       3  881250949
1     186      302       3  891717742
2      22      377       1  878887116
3     244       51       2  880606923
4     166      346       1  886397596

movies shape: (1682, 3)
   movieId              title release_date
0        1   Toy Story (1995)  01-Jan-1995
1        2   GoldenEye (1995)  01-Jan-1995
2        3  Four Rooms (1995)  01-Jan-1995
3        4  Get Shorty (1995)  01-Jan-1995
4        5     Copycat (1995)  01-Jan-1995


In [2]:
# 타임스탬프 → datetime 변환
ratings['datetime'] = pd.to_datetime(ratings['timestamp'], unit='s')

# 중복 제거 (같은 유저가 같은 영화에 여러 번 평점 → 최신 것만 유지)
before = len(ratings)
ratings = ratings.sort_values('timestamp').drop_duplicates(
    subset=['userId', 'movieId'], keep='last'
)
after = len(ratings)

# 평점 범위 확인 (1~5 외 이상치 제거)
ratings = ratings[ratings['rating'].between(1, 5)]

print(f"중복 제거: {before - after}개 제거됨")
print(f"정제 후 평점 수: {len(ratings):,}")
print(f"유저 수: {ratings['userId'].nunique()}")
print(f"영화 수: {ratings['movieId'].nunique()}")
print(f"\n기간: {ratings['datetime'].min()} ~ {ratings['datetime'].max()}")
print(f"\n결측치:\n{ratings.isnull().sum()}")

중복 제거: 0개 제거됨
정제 후 평점 수: 100,000
유저 수: 943
영화 수: 1682

기간: 1997-09-20 03:05:10 ~ 1998-04-22 23:10:38

결측치:
userId       0
movieId      0
rating       0
timestamp    0
datetime     0
dtype: int64


In [3]:
# 시간 순서로 정렬 후 80:20 분리
ratings_sorted = ratings.sort_values('timestamp').reset_index(drop=True)

split_idx = int(len(ratings_sorted) * 0.8)
train_df = ratings_sorted.iloc[:split_idx].copy()
test_df  = ratings_sorted.iloc[split_idx:].copy()

# 검증: test가 train보다 모두 최신이어야 함
assert test_df['timestamp'].min() >= train_df['timestamp'].max(), \
    "데이터 누수 발생! 확인 필요"

print(f"Train: {len(train_df):,}개")
print(f"  기간: {train_df['datetime'].min().date()} ~ {train_df['datetime'].max().date()}")
print(f"\nTest:  {len(test_df):,}개")
print(f"  기간: {test_df['datetime'].min().date()} ~ {test_df['datetime'].max().date()}")
print("\n시간 기준 분리 완료 — 데이터 누수 없음")

Train: 80,000개
  기간: 1997-09-20 ~ 1998-03-07

Test:  20,000개
  기간: 1998-03-07 ~ 1998-04-22

시간 기준 분리 완료 — 데이터 누수 없음


In [4]:
import os

os.makedirs('../data/processed', exist_ok=True)

train_df.to_csv('../data/processed/train.csv', index=False)
test_df.to_csv('../data/processed/test.csv', index=False)
ratings_sorted.to_csv('../data/processed/ratings_clean.csv', index=False)

print("저장 완료")
print(f"  train.csv       : {len(train_df):,}개")
print(f"  test.csv        : {len(test_df):,}개")
print(f"  ratings_clean.csv: {len(ratings_sorted):,}개")

저장 완료
  train.csv       : 80,000개
  test.csv        : 20,000개
  ratings_clean.csv: 100,000개
